# Einleitung

Dieses Projekt analysiert globale Wildfires anhand satellitenbasierter Daten des NASA FIRMS Systems.
Der Fokus liegt auf der Untersuchung der räumlichen Verteilung und der Intensität von Wildfires innerhalb eines kurzen Zeitraums (letzte 24 Stunden).

Der Datensatz enthält georeferenzierte Punktdaten, wobei jede Zeile ein detektiertes Wildfire repräsentiert.
Wichtige Variablen wie die geografische Lage (Latitude/Longitude), die Detektionssicherheit (confidence) sowie die Feuerintensität (Fire Radiative Power, FRP) werden genutzt, um die Verteilung und Eigenschaften der Brände zu analysieren.

Das Ziel dieses Projekts ist es, eine automatisierte Pipeline zu entwickeln, die Wildfires verarbeitet, relevante Informationen herausfiltert, eine gezielte Analyse ermöglicht und die Ergebnisse übersichtlich auf einer interaktiven Karte visualisiert.

Die Forschungsfrage, die mit diesem Projekt beantwortet werden soll, ist: *Wie sind die globalen Wildfire-Ereignisse der letzten 24 Stunden räumlich verteilt und welche Aussagen lassen sich über deren Intensität und Detektionssicherheit treffen?*

Zur Beantwortung dieser Forschungsfrage werden folgende Aspekte untersucht:

1. **Räumliche Verteilung**: Wie sind die Wildfire-Ereignisse weltweit verteilt?
2. **Intensität**: Wie verteilt sich die Feuerintensität (Fire Radiative Power, FRP)?
3. **Top-Ereignisse**: In welchen Ländern traten die die 10 stärksten Wildfires auf?
4. **Datenqualität**: Wie verteilt sich die Detektionssicherheit (confidence) über die erfassten Wildfires?
5. **Statistische Analyse**: Welche statistischen Aussagen lassen sich aus der Verteilung der Intensität ableiten?

### Schritt 1: Importe, Paths & Überblick verschaffen

In [ ]:
#Importe
from pathlib import Path
import sys
sys.path.append("..")

import pandas as pd
import geopandas as gpd
from scripts.spatial_tools import reverse_geocode_coordinates #Import meiner eigenen geocoder Funktion
from scripts.classification import classify_fire_intensity #Import meiner eigenen Klassifikationsmethdoe
import matplotlib.pyplot as plt

#Pfade
data_dir = Path("../data") #heisst "gehe eine Orderstruktur höher" und dann in "data"
raw_dir = data_dir / "raw" #wo der Rohdatenordner zu finden ist
processed_dir = data_dir / "processed" #wo der Ordner mit den verarbeiteten Rohdaten ist
output_dir = Path("../outputs") #wo die exportieren Karten und Grafiken hinkommen
csv_path = raw_dir / "SUOMI_VIIRS_C2_Global_24h.csv" #wo das CSV abgelget ist

#Rohdaten laden
df = pd.read_csv(csv_path)


In [ ]:
#Inhalte anzeigen & Überblick verschaffen
print(df.head())
print() #Zeilenumbruch
print(df.info())
print() #Zeilenumbruch

**Zusammenfassung Schritt 1:** _data.head()_ hat uns die ersten 5 Reihen ausgegeben. Daraus können wir ableiten, dass für die Forschungsfrage lediglich die Spalten "latitude", "longitude", "acq_date", "confidence" und "frp" relevant sind. Daher selektieren wir diese in Schritt 2. Die Header sollten ausserdem noch etwas verständlicher unbenannt werden. Dafür stütze ich mich auf die Informationen im User Guide de NASA (2018) (siehe Quellen am Schluss). Zudem hat uns _data.info()_ gezeigt, dass acq_date momentan noch ein String ist. Das müssen wir ebenfalls ändern.

### Schritt 2: Bereinigung der Rohdaten

In [ ]:
relevant_data = df[["latitude", "longitude", "acq_date", "confidence", "frp"]].copy() #relevante Spalten auswählen und eine echte Kopie herstellen vom original dataframe
print() #Zeilenumbruch
print(relevant_data.head()) #reduzierter Dataframe anzeigen
print() #Zeilenumbruch
print(relevant_data.isna().sum()) #überprüfen, ob fehlende Werte (NaN) bestehen
print() #Zeilenumbruch

#Header (Spaltenbezeichnunge) unbenennen:
relevant_data = relevant_data.rename(columns={
    "latitude": "lat",
    "longitude": "lon",
    "acq_date": "detection_date",
    "confidence": "detection_confidence",
    "frp": "fire_radiative_power"
})

print(relevant_data.head()) #neue Header anzeigen lassen resp. erste 5 Zeilen
print() #Zeilenumbruch

relevant_data["detection_date"] = pd.to_datetime(relevant_data["detection_date"]) #detection_date von string in ein Datum umwandeln
print(relevant_data.info()) #schauen ob der string erfolgreich umgewandelt wurde
print() #Zeilenumbruch

relevant_data.to_csv(processed_dir / "relevant_data.csv", index=False) #export der relevanten Daten als .csv in der "processed" Ordner



**Zusammenfassung Schritt 2:**
Wir machen von unserer Selektion des ursprünglichen Dataframes eine Kopie, um Komplikationen bei späteren Modifikationen an unserem Subset zu verhindern. *relevant_data.isna().sum()* hat ergeben, dass es bei unserem Datensatz keine fehlenden Werte gibt. Daher müssen wir an dieser Stelle nichts bereinigen. Das _detection_date_, das bisher noch als string erfasst war wurde jetzt in ein Datum umegwandelt.

Gemäss User Guide (NASA, 2018) sind die Header wie folgt zu interpretieren:
* detection_date = YYYYMMDD Erfassungsdatum in year (YYYY), month (MM) and day (DD)
* detection_confidence =  Branderkennungs-Konfidenz (“L”=low, “N”=nominal, “H”=high)
* fire_radiative_power Strahlungsleistung des Feuers in megawatt (Mass für thermische Intensität)

## Schritt 3: Datenanalyse

##### 3.1. Statistische Übersicht der Feuerintensitäten (FRP)

In [ ]:
print("Die statistische Übersicht der Feuerintensität (FRP) der letzten 24h:")
print(relevant_data["fire_radiative_power"].describe())

**Zusammenfassung Schritt 3.1.:** Die statistische Übersicht _(.describe())_ zeigt uns, dass es in den letzten 24h insgesamt 39787 Brandereignisse _(count)_ Weltweit gab. Davon weisen die meisten Waldbrände eine geringe Intensität (Werte in megawatt (MW)) auf. Der Median liegt bei 4.73 MW (Wert bei 50%), während der Durchschnitt (mean) durch wenige sehr starke Brände erhöht wird. Der hohe Maximalwert von 601.15 MW sowie die grosse Streuung (std) deuten auf einzelne extreme Ereignisse hin, während der Grossteil der Brände eher schwach ist.

##### 3.2. Histogramm: Verteilung der Feuerintensität

In [ ]:
relevant_data["fire_radiative_power"].hist(bins=50)

plt.xlabel("Fire Radiative Power (FRP)")
plt.ylabel("Anzahl Brände")
plt.title("Verteilung der Feuerintensität")

plt.savefig(output_dir / "histogram_fire_intensity.png", dpi=300, bbox_inches="tight") #export des histogramms als png in den output-Ordner
plt.show()

In [ ]:
relevant_data["intensity_class"] = relevant_data["fire_radiative_power"].apply(classify_fire_intensity)
counts = relevant_data["intensity_class"].value_counts()

print("Anzahl Brände pro Intensitätsklasse:\n")
print(counts)

**Zusammenfassung Schritt 3.2**: Das Histogramm zeigt deutlich, dass die Mehrheit der Wildfires eine geringe Fire Radiative Power (FRP) aufweist, während nur wenige Ereignisse sehr hohe Intensitätswerte erreichen. Diese rechtsschiefe Verteilung bestätigt die zuvor berechneten Kennzahlen, insbesondere den niedrigen Median im Vergleich zum Maximum, und verdeutlicht das Vorhandensein von Ausreissern mit sehr hoher Intensität. Dies wird weiter bestätigt durch die Zählung der Anzahl Brände pro Intensitätsklasse. Die Klassen wurde gem. https://experience.arcgis.com/experience/a0dcc4b8e8ab49b58a520f5acb983345/page/FIRE definiert.

##### 3.3. Detektionskonfidenz

In [ ]:
print("\n Konfidenzelevels detektierten Brandereignisse:")
print(relevant_data["detection_confidence"].value_counts()) #Zählung der Werte pro Konfidenzlevel

**Zusammenfassung Schritt 3.3:** Die meisten Wildfires wurden mit einem nominalen Konfidenzlevel detektiert. Im Vergleich zur Gesamtzahl weisen nur wenige Branddetektionen ein niedriges oder hohes Konfidenzniveau auf. Insgesamt deutet dies auf eine überwiegend zuverlässige Datengrundlage hin.

##### 3.4. Die 10 stärksten Brandereignisse und deren geografische Lokation

In [ ]:
strongest_wildfires = relevant_data.sort_values("fire_radiative_power", ascending = False).head(10).copy() #.copy(), weil der reduzierte df weiterverwendet wird

strongest_wildfires["country"] = reverse_geocode_coordinates(strongest_wildfires["lat"].astype(str) + ", " + strongest_wildfires["lon"].astype(str))

strongest_wildfires["country"] = strongest_wildfires["country"].str.split(",").str[-1].str.strip()


In [ ]:
print("Die 10 stärksten Brandereignisse nach Land:")
print(strongest_wildfires["country"].value_counts())

print("\n Die 10 stärksten Brandereignisse nach Intensität:")
print(strongest_wildfires[["country", "fire_radiative_power"]])


**Zusammenfassung Schritt 3.4.:** Die Analyse der 10 stärksten Wildfires _(.head(10))_ zeigt, dass besonders viele _(.value_counts())_ intensive Brände _(ascending = False)_ in den Vereinigten Staaten und in Russland registriert wurden. Die höchste gemessene Feuerintensität trat dabei in Russland mit einer Fire Radiative Power (FRP) von über 600 MW auf. Auch die Vereinigten Staaten weisen zwei sehr starke Ereignisse mit hohen FRP-Werten auf. Die Ergebnisse deuten darauf hin, dass sich die intensivsten Brände (300-749 MW) innerhalb des untersuchten 24-Stunden-Zeitraums räumlich auf wenige Regionen _(.value_counts())_ konzentrierten. Da die FRP den Energieausstoss eines Feuers beschreibt, weisen hohe Werte auf besonders intensive und energiereiche Brände hin.

### Schritt 4: Umwandlung GeoDataFrame und Erstellung interkative Karte

##### 4.1. Umwandlung DataFrame in GeoDataFrame mit aktiver Geometrie


In [ ]:
wildfires_gdf = gpd.GeoDataFrame(relevant_data,geometry=gpd.points_from_xy(relevant_data["lon"], relevant_data["lat"])) #DataFrame in GeoDataFrame umwandeln, mit einer Punkgeometrie
wildfires_gdf = wildfires_gdf.set_crs(epsg=4326) #Koordinatensystem setzen

wildfires_gdf.to_file(processed_dir / "wildfires.gpkg", driver="GPKG") #Export des GeoDataFrames als gpkg in den "processed" Ordner

In [ ]:
print(wildfires_gdf.head()) #zeigt uns die ersten 10 Zeilen des GeoDataFrames
print(type(wildfires_gdf)) #zeigt uns, dass wir jetzt einen GeoDataFrame haben
print(wildfires_gdf.crs.name, wildfires_gdf.crs) #zeigt uns den CRS Namen und den EPSG Code.

**Zusammenfassung Schritt 4.1:** In diesem Schritt wurde der ursprüngliche Pandas DataFrame (_relevant_data_) in einen GeoDataFrame umgewandelt (_gpd.GeoDataFrame_) mit aktiver Geometrie umgewandelt, indem aus den Längen- und Breitengraden Punktgeometrien (_gpd.points_from_xy(relevant_data["lon"], relevant_data["lat"])_) erstellt werden. Dadurch können die Wildfires räumlich dargestellt und weiterverarbeitet werden. Anschliessend wird dem GeoDataFrame ein Koordinatensystem (WGS84, EPSG:4326) zugewiesen (_.set_crs(epsg=4326)_), welches für die korrekte geografische Verortung und die Darstellung auf Karten notwendig ist. Das WGS 85 EPSG: 4326 ist der globale Standard für GPS Koordinaten und für unsere globale Karte geeignet, da die Brandereignisse als globale GPS-Koordinaten (Latitude/Longitude) vorliegen.

##### 4.2. Interaktive Kartenansicht der detektieren Waldbrände der letzten 24h


In [ ]:
wildfire_map = wildfires_gdf.explore(
    column="intensity_class",
    categorical=True,
    cmap="YlOrRd_r",
    popup=["fire_radiative_power", "intensity_class", "detection_confidence"],
    tooltip=["fire_radiative_power", "intensity_class"],
    legend=True,
    tiles="CartoDB dark_matter"
)
wildfire_map.save(output_dir / "wildfire_map.html") #export der Karte als html nach "outputs"
wildfire_map #Karte anzeigen

**Zusammenfassung Schritt 4.2:** Die interaktive Karte visualisiert die detektierten Brandereignisse, eingefärbt nach ihrer Intensitätsklasse. Beim Popup wird zudem die Detektionssicherheit angezeigt. Insgesamt zeigt sich, dass sich die Brandereignisse weltweit verteilt sind, mit einer tendenziellen Häufung im Bereich des Äquators und einer Abnahme in Richtung der Pole. Die Verteilung dürfte auf die klimatischen Bedingungen zurückzuführen sein.

### Schritt 5: Fazit

**Fazit:** Die Analyse der Wildfire-Daten der letzten 24 Stunden zeigt, dass Brandereignisse weltweit auftreten, jedoch nicht gleichmässig verteilt sind. Die statistische Auswertung zeigt zudem, dass die meisten Brände eine eher geringe Intensität aufweisen. Nur wenige Ereignisse erreichen sehr hohe Werte der Fire Radiative Power (FRP), was sich auch im Histogramm als rechtsschiefe Verteilung zeigt. Die stärksten Brandereignisse konzentrieren sich vor allem in den USA und in Russland, was darauf hindeutet, dass besonders intensive Brände nur in bestimmten Regionen auftreten. Auch die Analyse der Detektionssicherheit zeigt, dass der Grossteil der Brände mit einem nominalen Konfidenzlevel erfasst wurde. Insgesamt kann daher von einer soliden Datenbasis ausgegangen werden. Zusammenfassend konnte gezeigt werden, dass sich mit einer automatisierten Pipeline globale Wildfire-Daten gut analysieren lassen und sowohl räumliche Verteilungen als auch Unterschiede in der Intensität klar sichtbar werden.

### Quellen

##### Datenquelle Brandereignisse
VIIRS 375m / S-NPP (24h) https://firms.modaps.eosdis.nasa.gov/active_fire/ (abgerufen: 19.05.2026)

##### Quellen zur Dateninterpretation:
- ArcGIS Fire Radiative Power Erklärung: https://experience.arcgis.com/experience/a0dcc4b8e8ab49b58a520f5acb983345/page/FIRE
- CIRA / NOAA (2020): VIIRS Active Fire Quick Guide. https://rammb2.cira.colostate.edu/wp-content/uploads/2020/09/VIIRS_Active_Fire_Quick_Guide_v6.pdf
- NASA (2018): VIIRS Active Fire Product – User Guide, Version 1.4. https://lpdaac.usgs.gov/documents/427/VNP14_User_Guide_V1.pdf